# 01. BIOS Screen End-to-End Pipeline Ablation

Sources: `notebooks/01-bios-pipeline.md`, `src/lib/cv/biosPipeline.ts`, Hough (1962), Pizer et al. (1987), Otsu (1979), Bradley & Roth (2007), Smith (2007).  
Goal: measure the contribution of Hough/Homography/CLAHE/Adaptive Threshold/Connected Components/OCR and generate README-ready figures.

Input: `data/bios/ground-truth.csv` and `data/bios/<vendor>/*.jpg|png|jpeg`.

In [ ]:
from pathlib import Path
import itertools
import json
import math
import warnings

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'data'
DOCS_DIR = ROOT / 'docs'
ABLATION_DIR = DOCS_DIR / 'ablation-results'
PIPELINE_DIR = DOCS_DIR / 'cv-pipeline'
ABLATION_DIR.mkdir(parents=True, exist_ok=True)
PIPELINE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font='Malgun Gothic')

print(f'ROOT={ROOT}')
print(f'OpenCV={cv2.__version__}')


def save_placeholder_png(path, title, message='No labeled data yet'):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.text(0.5, 0.58, title, ha='center', va='center', fontsize=15, weight='bold')
    ax.text(0.5, 0.42, message, ha='center', va='center', fontsize=11)
    ax.set_axis_off()
    fig.tight_layout()
    fig.savefig(path, dpi=160)
    plt.close(fig)


try:
    import pytesseract
except Exception as exc:
    pytesseract = None
    warnings.warn(f'pytesseract import failed: {exc}')

BIOS_DIR = DATA_DIR / 'bios'
GT_PATH = BIOS_DIR / 'ground-truth.csv'
REQUIRED_COLUMNS = ['filename', 'vendor', 'angle_deg', 'lighting', 'top_menu_text', 'notes']

## Data Loading
`filename` is relative to `data/bios`, for example `ami/sample01.jpg`.

In [ ]:
def load_ground_truth(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=REQUIRED_COLUMNS)
    df = pd.read_csv(path)
    for col in REQUIRED_COLUMNS:
        if col not in df.columns:
            df[col] = np.nan
    return df[REQUIRED_COLUMNS]

truth = load_ground_truth(GT_PATH)
truth['image_path'] = truth['filename'].apply(lambda x: BIOS_DIR / str(x) if pd.notna(x) else None)
truth['exists'] = truth['image_path'].apply(lambda p: bool(p and p.exists()))
display(truth)
print(f'label rows: {len(truth)}, existing images: {truth.exists.sum() if len(truth) else 0}')

## Pipeline Functions
The functions mirror the TypeScript pipeline: Hough+Homography, CLAHE, Adaptive Gaussian Threshold, Connected Components, and Tesseract OCR.

In [ ]:
def read_gray(path: Path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'cannot read image: {path}')
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img, gray

def detect_screen_quad(gray, hough_vote=80, hough_min=50, hough_gap=10):
    edges = cv2.Canny(gray, 50, 150)
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, hough_vote, minLineLength=hough_min, maxLineGap=hough_gap)
    if lines is None:
        return None, edges, []
    h_lines, v_lines = [], []
    for [[x1, y1, x2, y2]] in lines:
        angle = abs(math.degrees(math.atan2(y2 - y1, x2 - x1)))
        if angle < 20:
            h_lines.append((x1, y1, x2, y2))
        elif angle > 70:
            v_lines.append((x1, y1, x2, y2))
    if len(h_lines) < 2 or len(v_lines) < 2:
        return None, edges, lines.reshape(-1, 4).tolist()
    h_lines = sorted(h_lines, key=lambda line: line[1])
    v_lines = sorted(v_lines, key=lambda line: line[0])
    top, bottom = h_lines[0], h_lines[-1]
    left, right = v_lines[0], v_lines[-1]
    tl, tr, br, bl = (left[0], top[1]), (right[0], top[1]), (right[0], bottom[1]), (left[0], bottom[1])
    h, w = gray.shape[:2]
    if abs(tr[0] - tl[0]) < w * 0.15 or abs(bl[1] - tl[1]) < h * 0.15:
        return None, edges, lines.reshape(-1, 4).tolist()
    return np.float32([tl, tr, br, bl]), edges, lines.reshape(-1, 4).tolist()

def rectify(gray, enabled=True):
    if not enabled:
        return gray, False, None, None
    quad, edges, lines = detect_screen_quad(gray)
    if quad is None:
        return gray, False, edges, lines
    h, w = gray.shape[:2]
    dst = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]])
    H = cv2.getPerspectiveTransform(quad, dst)
    return cv2.warpPerspective(gray, H, (w, h)), True, edges, lines

def apply_clahe(gray, enabled=True, clip=2.0, grid=8):
    if not enabled:
        return gray
    clahe = cv2.createCLAHE(clipLimit=float(clip), tileGridSize=(int(grid), int(grid)))
    return clahe.apply(gray)

def binarize(gray, enabled=True, block=11, C=2, method='gaussian'):
    if not enabled:
        return gray
    block = int(block) if int(block) % 2 == 1 else int(block) + 1
    adaptive = cv2.ADAPTIVE_THRESH_GAUSSIAN_C if method == 'gaussian' else cv2.ADAPTIVE_THRESH_MEAN_C
    return cv2.adaptiveThreshold(gray, 255, adaptive, cv2.THRESH_BINARY, block, int(C))

def filter_components(binary, enabled=True, min_area=50):
    if not enabled:
        return binary
    num, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    keep = np.zeros(binary.shape, dtype=np.uint8)
    for idx in range(1, num):
        if stats[idx, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == idx] = 255
    return keep

def ocr_text(image):
    if pytesseract is None:
        return ''
    return pytesseract.image_to_string(image, lang='eng', config='--psm 6').strip()

def levenshtein_ratio(a, b):
    a, b = str(a or '').lower().strip(), str(b or '').lower().strip()
    if not a and not b:
        return 1.0
    if not a or not b:
        return 0.0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return 1 - prev[-1] / max(len(a), len(b))

def run_pipeline(path, homography=True, clahe=True, threshold=True, components=True, clip=2.0, grid=8, block=11, C=2):
    color, gray = read_gray(path)
    warped, rectified, edges, lines = rectify(gray, homography)
    enhanced = apply_clahe(warped, clahe, clip=clip, grid=grid)
    binary = binarize(enhanced, threshold, block=block, C=C)
    cc = filter_components(binary, components)
    text = ocr_text(cc if threshold or components else enhanced)
    return {'color': color, 'gray': gray, 'warped': warped, 'rectified': rectified, 'enhanced': enhanced, 'binary': binary, 'components': cc, 'ocr_text': text, 'edges': edges, 'lines': lines}

## Stage Visualization
The first valid image is used to save `docs/cv-pipeline/bios-pipeline-stages.png` and the threshold comparison figure.

In [ ]:
valid_rows = truth[truth['exists']].copy()
if valid_rows.empty:
    print('No BIOS images found. Fill data/bios/<vendor> images and ground-truth.csv.')
    save_placeholder_png(PIPELINE_DIR / 'bios-pipeline-stages.png', 'BIOS pipeline stages')
    save_placeholder_png(PIPELINE_DIR / 'bios-threshold-comparison.png', 'BIOS threshold comparison')
else:
    row = valid_rows.iloc[0]
    out = run_pipeline(row.image_path)
    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    stages = [('Original', cv2.cvtColor(out['color'], cv2.COLOR_BGR2RGB)), ('Gray', out['gray']), ('Rectified', out['warped']), ('CLAHE', out['enhanced']), ('Binary/CC', out['components'])]
    for ax, (title, img) in zip(axes, stages):
        ax.imshow(img, cmap='gray' if img.ndim == 2 else None)
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(PIPELINE_DIR / 'bios-pipeline-stages.png', dpi=180)
    plt.show()

    otsu = cv2.threshold(out['enhanced'], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
    mean = binarize(out['enhanced'], True, method='mean')
    gauss = binarize(out['enhanced'], True, method='gaussian')
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, title, img in zip(axes, ['Otsu', 'Adaptive Mean', 'Adaptive Gaussian'], [otsu, mean, gauss]):
        ax.imshow(img, cmap='gray')
        ax.set_title(title)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(PIPELINE_DIR / 'bios-threshold-comparison.png', dpi=180)
    plt.show()

## 16-Combination Ablation
Accuracy is the fraction of samples with Levenshtein similarity >= 0.8. Results are saved to `docs/ablation-results/bios-pipeline-ablation.csv` and the compatibility filename `bios-ablation.csv`.

In [ ]:
rows = []
flags = ['homography', 'clahe', 'threshold', 'components']
for combo in tqdm(list(itertools.product([False, True], repeat=4)), desc='BIOS ablation'):
    params = dict(zip(flags, combo))
    sims = []
    for _, row in valid_rows.iterrows():
        try:
            result = run_pipeline(row.image_path, **params)
            sim = levenshtein_ratio(result['ocr_text'], row['top_menu_text'])
            sims.append(sim)
        except Exception as exc:
            warnings.warn(f'{row.filename} failed: {exc}')
    rows.append({**params, 'n': len(sims), 'mean_similarity': float(np.mean(sims)) if sims else np.nan, 'accuracy_at_0_8': float(np.mean(np.array(sims) >= 0.8)) if sims else np.nan})

ablation = pd.DataFrame(rows).sort_values(['accuracy_at_0_8', 'mean_similarity'], ascending=False)
ablation.to_csv(ABLATION_DIR / 'bios-pipeline-ablation.csv', index=False)
ablation.to_csv(ABLATION_DIR / 'bios-ablation.csv', index=False)
display(ablation)

if ablation['accuracy_at_0_8'].notna().any():
    labels = ablation[flags].astype(int).astype(str).agg(''.join, axis=1)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(x=labels, y=ablation['accuracy_at_0_8'], ax=ax, color='#4776b4')
    ax.set_xlabel('homography/clahe/threshold/components (0=off, 1=on)')
    ax.set_ylabel('Accuracy @ similarity >= 0.8')
    ax.set_ylim(0, 1)
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'bios-ablation.png', dpi=180)
    plt.show()
else:
    save_placeholder_png(ABLATION_DIR / 'bios-ablation.png', 'BIOS ablation')

## CLAHE / Adaptive Threshold Grid Search
The best row is the candidate update for TypeScript constants `CLAHE_CLIP`, `CLAHE_GRID`, `ADAPT_BLOCK`, and `ADAPT_C`.

In [ ]:
grid_rows = []
for clip, grid, block, C in tqdm(list(itertools.product([1, 2, 4, 8], [4, 8, 16], [9, 11, 15, 21], [0, 2, 4])), desc='CLAHE grid'):
    sims = []
    for _, row in valid_rows.iterrows():
        try:
            result = run_pipeline(row.image_path, clip=clip, grid=grid, block=block, C=C)
            sims.append(levenshtein_ratio(result['ocr_text'], row['top_menu_text']))
        except Exception as exc:
            warnings.warn(f'{row.filename} grid failed: {exc}')
    grid_rows.append({'clip': clip, 'grid': grid, 'block': block, 'C': C, 'n': len(sims), 'mean_similarity': float(np.mean(sims)) if sims else np.nan, 'accuracy_at_0_8': float(np.mean(np.array(sims) >= 0.8)) if sims else np.nan})

grid_df = pd.DataFrame(grid_rows).sort_values(['accuracy_at_0_8', 'mean_similarity'], ascending=False)
grid_df.to_csv(ABLATION_DIR / 'bios-clahe-gridsearch.csv', index=False)
display(grid_df.head(10))

if grid_df['accuracy_at_0_8'].notna().any():
    pivot = grid_df.groupby(['clip', 'grid'])['accuracy_at_0_8'].max().unstack()
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='viridis', ax=ax)
    ax.set_title('CLAHE grid search: best accuracy')
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'bios-clahe-gridsearch.png', dpi=180)
    plt.show()

    angle_df = []
    best = grid_df.iloc[0].to_dict()
    for angle, part in valid_rows.groupby('angle_deg'):
        sims = []
        for _, row in part.iterrows():
            result = run_pipeline(row.image_path, clip=best['clip'], grid=best['grid'], block=best['block'], C=best['C'])
            sims.append(levenshtein_ratio(result['ocr_text'], row['top_menu_text']))
        angle_df.append({'angle_deg': angle, 'accuracy_at_0_8': float(np.mean(np.array(sims) >= 0.8))})
    angle_df = pd.DataFrame(angle_df)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.lineplot(data=angle_df, x='angle_deg', y='accuracy_at_0_8', marker='o', ax=ax)
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(ABLATION_DIR / 'bios-angle-accuracy.png', dpi=180)
    plt.show()
else:
    save_placeholder_png(ABLATION_DIR / 'bios-clahe-gridsearch.png', 'BIOS CLAHE grid search')
    save_placeholder_png(ABLATION_DIR / 'bios-angle-accuracy.png', 'BIOS angle accuracy')
    print('No data available; TypeScript parameter recommendation is deferred.')

## Final Table
Use the first row below as the README and `src/lib/cv/biosPipeline.ts` parameter-update evidence.

In [ ]:
if 'grid_df' in globals() and not grid_df.empty:
    display(grid_df.head(1))
else:
    print('Fill ground-truth and images, then run all cells.')